# Test-Time Augmentation (TTA) Evaluation — Run 5 Model

Eval-only notebook to measure the impact of test-time augmentation on the
best model (Run 5: DINOv2-Large + UperNet, unfrozen backbone, 68.8% mIoU).

**TTA strategy:** Average predictions across 6 variants:
1. Original
2. Horizontal flip
3. rot90(k=0) — duplicate of original, gives 2/6 weight to canonical orientation
4. rot90(k=1) — 90° rotation
5. rot90(k=2) — 180° rotation
6. rot90(k=3) — 270° rotation

For each variant: transform image → forward pass → undo transform on logits →
accumulate. After all 6, average logits → argmax → final prediction.

**Expected improvement:** +1-3% mIoU based on literature for dense segmentation
with bilateral symmetry (brain anatomy).

**No training, no model changes, no code changes to `src/`.**

In [0]:
# Cell 0 — Install project wheel from DBFS

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

Processing /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.5.2
    Not uninstalling accelerate at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-551efbdd-af21-4e83-84f7-fa6d74cc53d8
    Can't uninstall 'accelerate'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Cell 1 — Configuration

import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- Run 5 model ----------
MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/unfrozen"

# ---------- Settings ----------
MAPPING_TYPE = "full"
CROP_SIZE = 518
SPLIT_STRATEGY = "interleaved"
NUM_TTA_VARIANTS = 6  # original + hflip + 4x rot90

print(f"Model:      {MODEL_DIR}")
print(f"Ontology:   {ONTOLOGY_PATH}")
print(f"Nissl 10\u00b5m: {NISSL_10_PATH}")
print(f"Crop size:  {CROP_SIZE}")
print(f"TTA variants: {NUM_TTA_VARIANTS}")

Model:      /dbfs/FileStore/allen_brain_data/models/unfrozen
Ontology:   /Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/ontology/structure_graph_1.json
Nissl 10µm: /dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd
Crop size:  518
TTA variants: 6


In [0]:
# Cell 2 — Load data + create validation dataset

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

# Load ontology and build full mapping
mapper = OntologyMapper(ONTOLOGY_PATH)
mapping = mapper.build_full_mapping()
NUM_LABELS = mapper.get_num_labels(mapping)
class_names = mapper.get_class_names(mapping)
print(f"Full mapping: {NUM_LABELS} classes (including background)")

# Load CCFv3 volumes
slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Loaded {slicer.num_slices} coronal slices")

# Create validation dataset (center crop, no augmentation)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=mapping,
    crop_size=CROP_SIZE, augment=False, split_strategy=SPLIT_STRATEGY,
)
print(f"Validation samples: {len(val_ds)}")

Full mapping: 1328 classes (including background)
Loaded 1320 coronal slices
Validation samples: 127


In [0]:
# Cell 3 — Load Run 5 model

import torch
from transformers import UperNetForSemanticSegmentation

model = UperNetForSemanticSegmentation.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded from: {MODEL_DIR}")
print(f"Total parameters:  {total_params:,}")
print(f"Device:            {device}")
print(f"Num labels:        {model.config.num_labels}")
assert model.config.num_labels == NUM_LABELS, (
    f"Model num_labels ({model.config.num_labels}) != mapping num_labels ({NUM_LABELS})"
)

# Sanity check forward pass
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE, device=device)
    out = model(pixel_values=dummy)
    print(f"Logits shape:      {out.logits.shape}")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)
    del out, dummy
torch.cuda.empty_cache()
print("Forward pass OK")

2026-03-15 16:51:16.233611: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773593476.249615    2668 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773593476.254574    2668 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773593476.268556    2668 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773593476.268570    2668 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773593476.268572    2668 computation_placer.cc:177] computation placer alr

[2026-03-15 16:51:22,688] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: No such file or directory
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Model loaded from: /dbfs/FileStore/allen_brain_data/models/unfrozen
Total parameters:  342,104,160
Device:            cuda
Num labels:        1328
Logits shape:      torch.Size([1, 1328, 518, 518])
Forward pass OK


In [0]:
# Cell 4 — TTA evaluation functions (inline per Databricks notebook convention)

import numpy as np
from tqdm.auto import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def normalize_image(image):
    """Convert uint8 grayscale (H, W) to normalized 3-channel float tensor (3, H, W)."""
    img_float = image.astype(np.float32) / 255.0
    img_3ch = np.stack([img_float, img_float, img_float], axis=0)
    for c in range(3):
        img_3ch[c] = (img_3ch[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img_3ch)


def forward_pass(model, image, device):
    """Single forward pass. image is uint8 (H, W). Returns logits (1, C, H, W) in fp32 on CPU."""
    pixel_values = normalize_image(image).unsqueeze(0).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        logits = model(pixel_values=pixel_values).logits
    return logits.float().cpu()


def tta_predict(model, image, device, num_labels):
    """Run 6-variant TTA on one image. Returns (H, W) int64 predictions.

    Variants: original, hflip, rot90(k=0,1,2,3).
    Logits are accumulated in fp32 on CPU; forward passes use fp16 on GPU.
    """
    h, w = image.shape[:2]
    sum_logits = torch.zeros(1, num_labels, h, w)

    # 1. Original
    sum_logits += forward_pass(model, image, device)

    # 2. Horizontal flip
    flipped = np.flip(image, axis=1).copy()
    logits = forward_pass(model, flipped, device)
    sum_logits += torch.flip(logits, dims=[3])  # undo flip on W axis
    del logits

    # 3-6. rot90 k=0,1,2,3
    for k in range(4):
        if k == 0:
            rotated = image
        else:
            rotated = np.rot90(image, k=k).copy()
        logits = forward_pass(model, rotated, device)
        if k != 0:
            logits = torch.rot90(logits, k=-k, dims=[2, 3])
        sum_logits += logits
        del logits

    preds = (sum_logits / 6.0).argmax(dim=1).squeeze(0).numpy().astype(np.int64)
    del sum_logits
    return preds


def evaluate_dataset(model, val_ds, device, num_labels, use_tta=False):
    """Evaluate model on validation set with optional TTA.

    Returns dict with mean_iou, overall_accuracy, class_ious, num_valid_classes.
    """
    all_preds = []
    all_labels = []
    desc = "TTA eval" if use_tta else "Baseline eval"

    for idx in tqdm(range(len(val_ds)), desc=desc):
        # Access raw data and apply deterministic preprocessing
        image, mask = val_ds._slices[idx]
        image, mask = val_ds._pad_if_needed(image, mask)
        image, mask = val_ds._center_crop(image, mask)

        if use_tta:
            preds = tta_predict(model, image, device, num_labels)
        else:
            logits = forward_pass(model, image, device)
            preds = logits.argmax(dim=1).squeeze(0).numpy().astype(np.int64)
            del logits

        all_preds.append(preds.ravel())
        all_labels.append(mask.ravel())

    # Concatenate and compute metrics
    preds_flat = np.concatenate(all_preds)
    labels_flat = np.concatenate(all_labels)

    total = len(labels_flat)
    correct = (preds_flat == labels_flat).sum()
    accuracy = float(correct) / total

    class_ious = {}
    valid_ious = []
    for cls in range(num_labels):
        pred_mask = preds_flat == cls
        label_mask = labels_flat == cls
        if label_mask.sum() == 0:
            class_ious[cls] = float("nan")
            continue
        intersection = (pred_mask & label_mask).sum()
        union = (pred_mask | label_mask).sum()
        if union == 0:
            class_ious[cls] = 0.0
        else:
            iou = float(intersection) / float(union)
            class_ious[cls] = iou
            valid_ious.append(iou)

    mean_iou = float(np.mean(valid_ious)) if valid_ious else 0.0

    return {
        "mean_iou": mean_iou,
        "overall_accuracy": accuracy,
        "class_ious": class_ious,
        "num_valid_classes": len(valid_ious),
    }


print("TTA evaluation functions defined.")
print(f"Strategy: {NUM_TTA_VARIANTS} variants (original + hflip + rot90 k=0,1,2,3)")
print(f"Validation samples: {len(val_ds)}")
print(f"Total forward passes (TTA): {len(val_ds) * NUM_TTA_VARIANTS}")

TTA evaluation functions defined.
Strategy: 6 variants (original + hflip + rot90 k=0,1,2,3)
Validation samples: 127
Total forward passes (TTA): 762


In [0]:
# Cell 5 — Run baseline + TTA evaluation

# Baseline: single forward pass (no TTA)
print("Running baseline evaluation (no TTA)...")
baseline_results = evaluate_dataset(model, val_ds, device, NUM_LABELS, use_tta=False)
print(f"Baseline mIoU:     {baseline_results['mean_iou']:.4f}")
print(f"Baseline accuracy: {baseline_results['overall_accuracy']:.4f}")
print(f"Valid classes:     {baseline_results['num_valid_classes']}")

torch.cuda.empty_cache()

# TTA: 6-variant averaged predictions
print(f"\nRunning TTA evaluation ({NUM_TTA_VARIANTS} variants)...")
tta_results = evaluate_dataset(model, val_ds, device, NUM_LABELS, use_tta=True)
print(f"TTA mIoU:          {tta_results['mean_iou']:.4f}")
print(f"TTA accuracy:      {tta_results['overall_accuracy']:.4f}")
print(f"Valid classes:     {tta_results['num_valid_classes']}")

# Comparison
delta_miou = tta_results["mean_iou"] - baseline_results["mean_iou"]
delta_acc = tta_results["overall_accuracy"] - baseline_results["overall_accuracy"]

print(f"\n{'='*65}")
print(f"{'Metric':<25} {'Baseline':>12} {'TTA':>12} {'Delta':>12}")
print(f"{'='*65}")
print(f"{'mIoU':<25} {baseline_results['mean_iou']:>12.4f} {tta_results['mean_iou']:>12.4f} {delta_miou:>+12.4f}")
print(f"{'Overall accuracy':<25} {baseline_results['overall_accuracy']:>12.4f} {tta_results['overall_accuracy']:>12.4f} {delta_acc:>+12.4f}")
print(f"{'Valid classes':<25} {baseline_results['num_valid_classes']:>12d} {tta_results['num_valid_classes']:>12d}")
print(f"{'='*65}")

# Reference: documented Run 5 mIoU from Trainer eval = 68.8%
print(f"\nDocumented Run 5 mIoU (Trainer eval): 0.6880")
print(f"Baseline mIoU (this notebook):         {baseline_results['mean_iou']:.4f}")
print(f"TTA mIoU:                              {tta_results['mean_iou']:.4f}")
if abs(baseline_results['mean_iou'] - 0.6880) > 0.005:
    print(f"\nNOTE: Baseline differs from documented 68.8% by "
          f"{baseline_results['mean_iou'] - 0.6880:+.4f}. "
          f"Small differences expected due to center-crop determinism.")

Running baseline evaluation (no TTA)...


Baseline eval:   0%|          | 0/127 [00:00<?, ?it/s]

Baseline mIoU:     0.6844
Baseline accuracy: 0.9212
Valid classes:     504

Running TTA evaluation (6 variants)...


TTA eval:   0%|          | 0/127 [00:00<?, ?it/s]

TTA mIoU:          0.4439
TTA accuracy:      0.8435
Valid classes:     504

Metric                        Baseline          TTA        Delta
mIoU                            0.6844       0.4439      -0.2405
Overall accuracy                0.9212       0.8435      -0.0777
Valid classes                      504          504

Documented Run 5 mIoU (Trainer eval): 0.6880
Baseline mIoU (this notebook):         0.6844
TTA mIoU:                              0.4439


In [0]:
# Cell 6 — Per-class IoU analysis: TTA vs baseline

baseline_ious = baseline_results["class_ious"]
tta_ious = tta_results["class_ious"]

# Compute per-class delta
class_deltas = {}
for cls in range(NUM_LABELS):
    b_iou = baseline_ious.get(cls, float("nan"))
    t_iou = tta_ious.get(cls, float("nan"))
    if not np.isnan(b_iou) and not np.isnan(t_iou):
        class_deltas[cls] = t_iou - b_iou

# Top 10 classes by TTA IoU
sorted_tta = sorted(
    [(cls, iou) for cls, iou in tta_ious.items() if not np.isnan(iou)],
    key=lambda x: x[1], reverse=True,
)

print("Top 10 classes by TTA IoU:")
print(f"{'Class':>6} {'Name':<45} {'Baseline':>9} {'TTA':>9} {'Delta':>8}")
print("-" * 80)
for cls, iou in sorted_tta[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    b = baseline_ious.get(cls, float("nan"))
    d = class_deltas.get(cls, float("nan"))
    print(f"{cls:>6} {name:<45} {b:>9.4f} {iou:>9.4f} {d:>+8.4f}")

# Bottom 10 (non-zero) classes by TTA IoU
nonzero_tta = [(cls, iou) for cls, iou in sorted_tta if iou > 0]
print(f"\nBottom 10 classes by TTA IoU (non-zero):")
print(f"{'Class':>6} {'Name':<45} {'Baseline':>9} {'TTA':>9} {'Delta':>8}")
print("-" * 80)
for cls, iou in nonzero_tta[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    b = baseline_ious.get(cls, float("nan"))
    d = class_deltas.get(cls, float("nan"))
    print(f"{cls:>6} {name:<45} {b:>9.4f} {iou:>9.4f} {d:>+8.4f}")

# Top 10 classes by TTA improvement
sorted_deltas = sorted(class_deltas.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by TTA improvement:")
print(f"{'Class':>6} {'Name':<45} {'Baseline':>9} {'TTA':>9} {'Delta':>8}")
print("-" * 80)
for cls, delta in sorted_deltas[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"{cls:>6} {name:<45} {baseline_ious[cls]:>9.4f} {tta_ious[cls]:>9.4f} {delta:>+8.4f}")

# Top 10 classes by TTA regression
print(f"\nTop 10 classes by TTA regression:")
print(f"{'Class':>6} {'Name':<45} {'Baseline':>9} {'TTA':>9} {'Delta':>8}")
print("-" * 80)
for cls, delta in sorted_deltas[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"{cls:>6} {name:<45} {baseline_ious[cls]:>9.4f} {tta_ious[cls]:>9.4f} {delta:>+8.4f}")

# Summary statistics
deltas_arr = np.array(list(class_deltas.values()))
print(f"\nPer-class delta summary ({len(deltas_arr)} classes with valid IoU):")
print(f"  Mean delta:      {deltas_arr.mean():+.4f}")
print(f"  Median delta:    {np.median(deltas_arr):+.4f}")
print(f"  Std delta:       {deltas_arr.std():.4f}")
print(f"  Classes improved:  {(deltas_arr > 0).sum()} / {len(deltas_arr)}")
print(f"  Classes regressed: {(deltas_arr < 0).sum()} / {len(deltas_arr)}")
print(f"  Classes unchanged: {(deltas_arr == 0).sum()} / {len(deltas_arr)}")

Top 10 classes by TTA IoU:
 Class Name                                           Baseline       TTA    Delta
--------------------------------------------------------------------------------
   662 Caudoputamen                                     0.9717    0.9497  -0.0220
   497 Main olfactory bulb                              0.9672    0.9242  -0.0431
   958 Nodulus (X)                                      0.9536    0.9135  -0.0400
  1080 Lobules IV-V                                     0.9322    0.9063  -0.0258
    51 Nucleus accumbens                                0.9312    0.8976  -0.0335
   947 Uvula (IX)                                       0.9362    0.8938  -0.0424
   974 Lobule III                                       0.9395    0.8936  -0.0459
   372 Field CA1                                        0.9341    0.8922  -0.0419
   419 Spinal nucleus of the trigeminal, caudal part    0.9150    0.8912  -0.0238
     0 Background                                       0.9640    0.8888